# Phase 2 : Behavioral Exploration

## Goal

Test whether asking Qwen3-4B to attend to or report its current response tendency changes its observable behavior before looking at internal activations.

This phase tests for a behavioral effect only, not a causal internal mechanism.

## Approach

1. Create three different A/B decision tasks.
2. Run each task under Control, Neutral, Silent Self-Attention, and Report conditions.
3. Compare the model's final A/B choices.
4. Compare the A-vs-B next-token logit margins to detect smaller shifts in response tendency.

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = "/workspace/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda"
)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model: Qwen/Qwen3-4B
Device: cuda:0
Dtype: torch.bfloat16


In [3]:
import sys
import importlib.util

print("Python executable:", sys.executable)
print("Transformers found:", importlib.util.find_spec("transformers"))

Python executable: /usr/local/bin/python
Transformers found: ModuleSpec(name='transformers', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7ba6ae2ed9d0>, origin='/usr/local/lib/python3.12/dist-packages/transformers/__init__.py', submodule_search_locations=[PosixPath('/usr/local/lib/python3.12/dist-packages/transformers')])


In [4]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = "/workspace/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda"
)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model: Qwen/Qwen3-4B
Device: cuda:0
Dtype: torch.bfloat16


In [5]:
prompt = "The Eiffel Tower is located in"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)

The Eiffel Tower is located in which city? The Eiffel Tower is located


In [6]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

In [7]:
prompt = "The Eiffel Tower is located in"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)

The Eiffel Tower is located in which city? The Eiffel Tower is located


In [8]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.8.0+cu128
Transformers: 5.16.1
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [9]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_HOME"] = "/workspace/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda"
)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model: Qwen/Qwen3-4B
Device: cuda:0
Dtype: torch.bfloat16


In [10]:
control_prompt = """
A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
"""

print(control_prompt)


A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.



In [11]:
inputs = tokenizer(
    control_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)


A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
Answer: A
Answer


In [12]:
neutral_prompt = """
Before answering, carefully read all of the options presented below.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
"""

print(neutral_prompt)


Before answering, carefully read all of the options presented below.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.



In [13]:
inputs = tokenizer(
    neutral_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)


Before answering, carefully read all of the options presented below.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
Answer: A
Okay


In [14]:
silent_prompt = """
Before answering, silently attend to your current response tendency.
Do not report or describe that tendency.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
"""

print(silent_prompt)


Before answering, silently attend to your current response tendency.
Do not report or describe that tendency.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.



In [15]:
inputs = tokenizer(
    silent_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)


Before answering, silently attend to your current response tendency.
Do not report or describe that tendency.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
Answer: A
Okay


In [16]:
report_prompt = """
Before answering, attend to your current response tendency.
First report which option you currently lean toward, then give your final answer.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Report your current response tendency, then give your final answer as A or B.
"""

print(report_prompt)


Before answering, attend to your current response tendency.
First report which option you currently lean toward, then give your final answer.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Report your current response tendency, then give your final answer as A or B.



In [17]:
inputs = tokenizer(
    report_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)


Before answering, attend to your current response tendency.
First report which option you currently lean toward, then give your final answer.

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Report your current response tendency, then give your final answer as A or B.
Okay, so I need to figure out whether the startup should launch now with a simpler product and get user feedback early (option A) or delay the launch by three months to make the product more polished (option B). Let me think through the pros and cons of each option.

Starting with option A: Launching now with a simpler product. The main advantage here is that they can get user feedback early. That feedback can help them understand what users like and what they don't, which can guide


In [18]:
report_prompt_v2 = """
A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Before giving your final answer, attend to your current response tendency.

Do not explain or reason about the options.
Report only:
Tendency: A or B
Final: A or B
"""

print(report_prompt_v2)


A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Before giving your final answer, attend to your current response tendency.

Do not explain or reason about the options.
Report only:
Tendency: A or B
Final: A or B



In [19]:
inputs = tokenizer(
    report_prompt_v2,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(generated_text)


A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Before giving your final answer, attend to your current response tendency.

Do not explain or reason about the options.
Report only:
Tendency: A or B
Final: A or B
Okay, let's see. The startup has two options: launch now with a simpler product and get


In [20]:
print(tokenizer.chat_template[:2000])

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set index = (messages|length - 

In [21]:
messages = [
    {"role": "user", "content": control_prompt}
]

control_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(control_text)

<|im_start|>user

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
<|im_end|>
<|im_start|>assistant



In [22]:
inputs = tokenizer(
    control_text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("MODEL ANSWER:")
print(repr(answer))

MODEL ANSWER:
'<think>\nOkay, the user is asking which option'


In [23]:
messages = [
    {"role": "user", "content": control_prompt}
]

control_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

print(control_text_no_think)

<|im_start|>user

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
<|im_end|>
<|im_start|>assistant
<think>

</think>




In [24]:
messages = [
    {"role": "user", "content": control_prompt}
]

control_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

print(control_text_no_think)

<|im_start|>user

A startup has two options:

A) Launch now with a simpler product and get user feedback early.
B) Delay launch by three months to make the product more polished.

Which option would you choose?
Answer only A or B.
<|im_end|>
<|im_start|>assistant
<think>

</think>




In [25]:
inputs = tokenizer(
    control_text_no_think,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("CONTROL ANSWER:")
print(repr(answer))

CONTROL ANSWER:
'A'


In [26]:
messages = [
    {"role": "user", "content": neutral_prompt}
]

neutral_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    neutral_text_no_think,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

neutral_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("NEUTRAL ANSWER:")
print(repr(neutral_answer))

NEUTRAL ANSWER:
'A'


In [27]:
messages = [
    {"role": "user", "content": neutral_prompt}
]

neutral_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    neutral_text_no_think,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

neutral_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("NEUTRAL ANSWER:")
print(repr(neutral_answer))

NEUTRAL ANSWER:
'A'


In [28]:
messages = [
    {"role": "user", "content": neutral_prompt}
]

neutral_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    neutral_text_no_think,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

neutral_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("NEUTRAL ANSWER:")
print(repr(neutral_answer))

NEUTRAL ANSWER:
'A'


In [29]:
messages = [
    {"role": "user", "content": silent_prompt}
]

silent_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    silent_text_no_think,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

silent_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("SILENT ANSWER:")
print(repr(silent_answer))

SILENT ANSWER:
'A'


In [30]:
messages = [
    {"role": "user", "content": report_prompt_v2}
]

report_text_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    report_text_no_think,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

report_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("REPORT ANSWER:")
print(repr(report_answer))

REPORT ANSWER:
'Tendency: A  \nFinal: A'


In [31]:
task2_base = """
You need to make a decision.

A) Submit an analysis now with 90% confidence that it is correct.
B) Spend another day checking the analysis and aim for 98% confidence.

Which option would you choose?
Answer only A or B.
"""

print(task2_base)


You need to make a decision.

A) Submit an analysis now with 90% confidence that it is correct.
B) Spend another day checking the analysis and aim for 98% confidence.

Which option would you choose?
Answer only A or B.



In [32]:
messages = [
    {"role": "user", "content": task2_base}
]

task2_control_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    task2_control_text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task2_control_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 2 — CONTROL:")
print(repr(task2_control_answer))

TASK 2 — CONTROL:
'A'


In [33]:
messages = [
    {"role": "user", "content": task2_base}
]

task2_control_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    task2_control_text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task2_control_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 2 — CONTROL:")
print(repr(task2_control_answer))


TASK 2 — CONTROL:
'A'


In [34]:
task2_neutral = """
Before answering, carefully read all of the options presented below.

You need to make a decision.

A) Submit an analysis now with 90% confidence that it is correct.
B) Spend another day checking the analysis and aim for 98% confidence.

Which option would you choose?
Answer only A or B.
"""

messages = [
    {"role": "user", "content": task2_neutral}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task2_neutral_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 2 — NEUTRAL:")
print(repr(task2_neutral_answer))

TASK 2 — NEUTRAL:
'A'


In [35]:
task2_silent = """
Before answering, silently attend to your current response tendency.
Do not report or describe that tendency.

You need to make a decision.

A) Submit an analysis now with 90% confidence that it is correct.
B) Spend another day checking the analysis and aim for 98% confidence.

Which option would you choose?
Answer only A or B.
"""

messages = [
    {"role": "user", "content": task2_silent}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task2_silent_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 2 — SILENT:")
print(repr(task2_silent_answer))

TASK 2 — SILENT:
'A'


In [36]:
task2_report = """
You need to make a decision.

A) Submit an analysis now with 90% confidence that it is correct.
B) Spend another day checking the analysis and aim for 98% confidence.

Before giving your final answer, attend to your current response tendency.

Do not explain or reason about the options.
Report only:
Tendency: A or B
Final: A or B
"""

messages = [
    {"role": "user", "content": task2_report}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task2_report_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 2 — REPORT:")
print(repr(task2_report_answer))

TASK 2 — REPORT:
'Tendency: A  \nFinal: A'


In [37]:
task3_base = """
You need to make a decision.

A) Choose a candidate with stronger technical skills but weaker communication skills.
B) Choose a candidate with slightly weaker technical skills but stronger communication skills.

Which option would you choose?
Answer only A or B.
"""

print(task3_base)


You need to make a decision.

A) Choose a candidate with stronger technical skills but weaker communication skills.
B) Choose a candidate with slightly weaker technical skills but stronger communication skills.

Which option would you choose?
Answer only A or B.



In [38]:
messages = [
    {"role": "user", "content": task3_base}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task3_control_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 3 — CONTROL:")
print(repr(task3_control_answer))

TASK 3 — CONTROL:
'B'


In [39]:
task3_neutral = """
Before answering, carefully read all of the options presented below.

You need to make a decision.

A) Choose a candidate with stronger technical skills but weaker communication skills.
B) Choose a candidate with slightly weaker technical skills but stronger communication skills.

Which option would you choose?
Answer only A or B.
"""

messages = [{"role": "user", "content": task3_neutral}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task3_neutral_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 3 — NEUTRAL:")
print(repr(task3_neutral_answer))

TASK 3 — NEUTRAL:
'B'


In [40]:
task3_silent = """
Before answering, silently attend to your current response tendency.
Do not report or describe that tendency.

You need to make a decision.

A) Choose a candidate with stronger technical skills but weaker communication skills.
B) Choose a candidate with slightly weaker technical skills but stronger communication skills.

Which option would you choose?
Answer only A or B.
"""

messages = [{"role": "user", "content": task3_silent}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task3_silent_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 3 — SILENT:")
print(repr(task3_silent_answer))

TASK 3 — SILENT:
'B'


In [41]:
task3_report = """
You need to make a decision.

A) Choose a candidate with stronger technical skills but weaker communication skills.
B) Choose a candidate with slightly weaker technical skills but stronger communication skills.

Before giving your final answer, attend to your current response tendency.

Do not explain or reason about the options.
Report only:
Tendency: A or B
Final: A or B
"""

messages = [{"role": "user", "content": task3_report}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=False
    )

new_tokens = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

task3_report_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("TASK 3 — REPORT:")
print(repr(task3_report_answer))

TASK 3 — REPORT:
'Tendency: B  \nFinal: B'


In [42]:
for text in ["A", "B", " A", " B"]:
    ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    tokens = tokenizer.convert_ids_to_tokens(ids)

    print(
        repr(text),
        "-> IDs:", ids,
        "Tokens:", tokens
    )

'A' -> IDs: [32] Tokens: ['A']
'B' -> IDs: [33] Tokens: ['B']
' A' -> IDs: [362] Tokens: ['ĠA']
' B' -> IDs: [425] Tokens: ['ĠB']


In [43]:
messages = [{"role": "user", "content": control_prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1,
        do_sample=False
    )

first_generated_id = output_ids[0, -1].item()
first_generated_token = tokenizer.convert_ids_to_tokens(
    [first_generated_id]
)[0]

first_generated_text = tokenizer.decode(
    [first_generated_id]
)

print("Generated token ID:", first_generated_id)
print("Generated token:", repr(first_generated_token))
print("Decoded text:", repr(first_generated_text))

Generated token ID: 32
Generated token: 'A'
Decoded text: 'A'


In [44]:
A_ID = 32
B_ID = 33

with torch.no_grad():
    outputs = model(**inputs)

last_logits = outputs.logits[0, -1, :].float()

logit_A = last_logits[A_ID].item()
logit_B = last_logits[B_ID].item()

margin_A_minus_B = logit_A - logit_B

print("Logit A:", logit_A)
print("Logit B:", logit_B)
print("A - B margin:", margin_A_minus_B)

Logit A: 54.75
Logit B: 31.375
A - B margin: 23.375


In [45]:
messages = [{"role": "user", "content": neutral_prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs_neutral = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_neutral = model(**inputs_neutral)

last_logits_neutral = outputs_neutral.logits[0, -1, :].float()

logit_A_neutral = last_logits_neutral[A_ID].item()
logit_B_neutral = last_logits_neutral[B_ID].item()

margin_neutral = logit_A_neutral - logit_B_neutral

print("TASK 1 — NEUTRAL")
print("Logit A:", logit_A_neutral)
print("Logit B:", logit_B_neutral)
print("A - B margin:", margin_neutral)

TASK 1 — NEUTRAL
Logit A: 55.5
Logit B: 29.25
A - B margin: 26.25


In [46]:
messages = [{"role": "user", "content": silent_prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs_silent = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_silent = model(**inputs_silent)

last_logits_silent = outputs_silent.logits[0, -1, :].float()

logit_A_silent = last_logits_silent[A_ID].item()
logit_B_silent = last_logits_silent[B_ID].item()

margin_silent = logit_A_silent - logit_B_silent

print("TASK 1 — SILENT")
print("Logit A:", logit_A_silent)
print("Logit B:", logit_B_silent)
print("A - B margin:", margin_silent)

TASK 1 — SILENT
Logit A: 55.75
Logit B: 41.5
A - B margin: 14.25


## A/B Logit Margins

The final answer stayed the same across conditions within each task. To check for smaller shifts in response tendency, I next compare the A-vs-B next-token logit margin.

In [47]:
def get_ab_margin(prompt):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs_local = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs_local = model(**inputs_local)

    last_logits = outputs_local.logits[0, -1, :].float()

    logit_A = last_logits[A_ID].item()
    logit_B = last_logits[B_ID].item()
    margin = logit_A - logit_B

    return {
        "logit_A": logit_A,
        "logit_B": logit_B,
        "margin": margin
    }

In [48]:
task2_margins = {
    "Control": get_ab_margin(task2_base),
    "Neutral": get_ab_margin(task2_neutral),
    "Silent": get_ab_margin(task2_silent)
}

for condition, result in task2_margins.items():
    print(
        condition,
        "A:", result["logit_A"],
        "B:", result["logit_B"],
        "Margin:", result["margin"]
    )

Control A: 47.75 B: 43.5 Margin: 4.25
Neutral A: 51.0 B: 44.5 Margin: 6.5
Silent A: 50.0 B: 43.75 Margin: 6.25


In [49]:
task3_margins = {
    "Control": get_ab_margin(task3_base),
    "Neutral": get_ab_margin(task3_neutral),
    "Silent": get_ab_margin(task3_silent)
}

for condition, result in task3_margins.items():
    print(
        condition,
        "A:", result["logit_A"],
        "B:", result["logit_B"],
        "Margin:", result["margin"]
    )

Control A: 33.25 B: 52.25 Margin: -19.0
Neutral A: 29.875 B: 50.5 Margin: -20.625
Silent A: 34.75 B: 49.75 Margin: -15.0


In [50]:
import pandas as pd

phase2_results = pd.DataFrame({
    "Task": ["Task 1", "Task 2", "Task 3"],

    "Control": [
        23.375,
        4.250,
        -19.000
    ],

    "Neutral": [
        26.250,
        6.500,
        -20.625
    ],

    "Silent": [
        14.250,
        6.250,
        -15.000
    ]
})

phase2_results["Δ |margin| Silent−Control"] = (
    phase2_results["Silent"]
    - phase2_results["Control"]
)

phase2_results["Δ |margin| Silent−Neutral"] = (
    phase2_results["Silent"]
    - phase2_results["Neutral"]
)

phase2_results

,Task,Control,Neutral,Silent,Δ |margin| Silent−Control,Δ |margin| Silent−Neutral
0,Task 1,23.375,26.250,14.25,-9.125,-12.000
1,Task 2,4.250,6.500,6.25,2.000,-0.250
2,Task 3,-19.000,-20.625,-15.00,4.000,5.625


In [51]:
phase2_results["|Control|"] = phase2_results["Control"].abs()
phase2_results["|Neutral|"] = phase2_results["Neutral"].abs()
phase2_results["|Silent|"] = phase2_results["Silent"].abs()

phase2_results["Δ |margin| Silent−Control"] = (
    phase2_results["|Silent|"]
    - phase2_results["|Control|"]
)

phase2_results["Δ |margin| Silent−Neutral"] = (
    phase2_results["|Silent|"]
    - phase2_results["|Neutral|"]
)

phase2_results[
    [
        "Task",
        "|Control|",
        "|Neutral|",
        "|Silent|",
        "Δ |margin| Silent−Control",
        "Δ |margin| Silent−Neutral",
    ]
]

,Task,|Control|,|Neutral|,|Silent|,Δ |margin| Silent−Control,Δ |margin| Silent−Neutral
0,Task 1,23.375,26.250,14.25,-9.125,-12.000
1,Task 2,4.250,6.500,6.25,2.000,-0.250
2,Task 3,19.000,20.625,15.00,-4.000,-5.625


## Result

The final A/B choice did not change across conditions in any of the three tasks, but the A-vs-B next-token logit margins did.

Under Silent Self-Attention:

- **Task 1:** +23.375 → +14.250, weakening a strong preference for A.
- **Task 2:** +4.250 → +6.250, slightly strengthening the preference for A.
- **Task 3:** -19.000 → -15.000, weakening the preference for B.

The direction and size of these shifts were not consistent across tasks. I therefore do not treat this pilot as evidence for a systematic Observer Effect.


## Conclusion

The main limitation of Phase 2 was not only the small sample size. The tasks also started from different behavioral regimes and differed in content, which made them difficult to compare as measurements of the same behavioral construct.

Adding more heterogeneous tasks would increase N, but it would not solve the underlying problems of comparability and construct validity.

I therefore narrowed the assay to a controlled factual-sycophancy setting, where the same behavioral quantity can be measured across more comparable items. This should make any observed differences easier to interpret.

**The research question remains the same: Does directing a language model's attention toward its own current response tendency alter its internal state, its response tendency, or the downstream computation that leads to the final answer?**

**What changes is the assay used to test it.**